
# Notebook 39 (Claude) — Admissibility and Conflict Adjudication

**Zero simulations. Zero posterior draws.** Everything here is computed from artifacts
already on disk.

The Cursor sprint's Notebook 39 asked one admissibility question — *was this metric used
to fit or to select?* — and answered it correctly. This notebook adopts that answer and
asks the second question it skipped: **can this metric's decision rule return an
unfavorable answer at all?** A rule whose admissible set covers everything the simulator
can produce is not a test, and a FAVORABLE from it carries no information.

Phases:

| | |
|---|---|
| 0 | immutables attestation — `NO_GO`, sealed closed, `E2_ref` frozen |
| 1 | lineage reproduction — adopt the Cursor classification |
| 2 | **power precheck** — the new gate |
| 3 | rate-precision audit — can the family resolve its own effect? |
| 4 | support-conflict adjudication |
| 5 | failure-wording ledger |
| 6 | handoff to `claude_40` |

Plan of record: `docs/NOTEBOOK_39_41_CLAUDE_PLAN.md`.


In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path.cwd().resolve().parents[0] / "src"
if not SRC.is_dir():
    SRC = Path.cwd().resolve().parents[1] / "S4_sbi" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sleep_sbi import figure10_closure_claude_common as common
from sleep_sbi.figure10_protocol import PROJECT_ROOT

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def show(payload, keys=None):
    """Print a JSON-ish payload compactly, optionally restricted to some keys."""
    if keys is not None:
        payload = {k: payload[k] for k in keys if k in payload}
    print(json.dumps(payload, indent=2, ensure_ascii=False, default=str)[:4000])


print("python  :", sys.version.split()[0])
print("env     :", Path(sys.executable).parent.name)
print("src     :", SRC)


python  : 3.10.20
env     : neurolib
src     : D:\Year3_Mao_Projects\sleep_loop\S4_sbi\src


In [2]:

from sleep_sbi import figure10_notebook39_claude as nb39

paths = common.claude_dirs(39)
print("artifact root:", common.rel(paths["root"]))


artifact root: S4_sbi/results/figure10_8d_7d/artifacts/notebook_39_claude



## Phase 0 — Immutables

Nothing downstream may move the global verdict, the sealed bank, or the frozen estimator
choice. The run refuses to start if any of them has moved.


In [3]:

immutables = common.assert_immutables()
seeds = common.assert_seed_disjointness()

show(immutables, ["global_uncertainty_status", "nb39_handoff_mode", "e2_ref",
                  "seven_d_development_status", "eight_d_development_status",
                  "sealed_opened", "nb38_decision_sha256"])
print("\nseed block base", seeds["base"], "| cursor block base", seeds["cursor_base"],
      "| overlap", seeds["overlap"])


{
  "global_uncertainty_status": "NO_GO",
  "nb39_handoff_mode": "negative_evidence_only",
  "e2_ref": {
    "7d": "member_4",
    "8d": "member_3"
  },
  "seven_d_development_status": "DEV_FAIL",
  "eight_d_development_status": "DEV_FAIL",
  "sealed_opened": false,
  "nb38_decision_sha256": "c67ea6c3d110cd1bacf20020050c93a93a7375eb699b0bb8a8bc819d014ebd0f"
}

seed block base 9851000 | cursor block base 9841000 | overlap []



## Phase 1 — Lineage, adopted

Both physiology families really are objective-disjoint and really were not used to select
`E2_ref`. We take that as given and record that we are adopting it, so that the
disagreement with the Cursor sprint sits in exactly one place.


In [4]:

lineage = nb39.phase1_lineage_reproduction(paths)
display(pd.DataFrame(lineage["families"]))
print(lineage["verdict"], "|", lineage["note"])


,family,lineage_class,de_objective,influenced_selection,adopted_from
0,so_waveform,decision_held_out_descriptive,NOT_USED,False,cursor_nb39_heldout_metric_manifest
1,spindle_density,decision_held_out_descriptive,NOT_USED,False,cursor_nb39_heldout_metric_manifest


ADOPTED_WITHOUT_CHANGE | Lineage (was this metric used to fit or to select?) and discriminative power (can this metric's rule return an unfavorable answer?) are independent admissibility questions. The Cursor audit answered the first and skipped the second.



## Phase 2 — The power precheck

For each family: what is the set of outcomes the frozen rule calls FAVORABLE, and what is
the set of outcomes this model family can actually produce? If the second sits inside the
first, the rule cannot fail.

This needs only the historical validator tables, which predate the entire SBI pipeline —
so the gate is executable *before* any candidate is drawn.


In [5]:

power = nb39.phase2_power_precheck(paths)

sp = power["families"]["spindle_density"]
print("spindle_density")
print("  rule                :", sp["rule"])
print("  d_real              : %.4f /min" % sp["d_real"])
print("  d_V7 (historical)   : %.4f /min" % sp["d_v7_historical"])
print("  FAVORABLE set       : [%.4f, %.4f]" % tuple(sp["admissible_interval"]))
print("  attainable by model : [%.4f, %.4f]" % tuple(sp["attainable_interval_historical"]))
print("  unfavorable reachable:", sp["unfavorable_reachable"])
print("  VERDICT             :", sp["verdict"])
print()
print(sp["argument"])


spindle_density
  rule                : FAVORABLE iff |d_cand - d_real| <= |d_V7 - d_real|
  d_real              : 2.9580 /min
  d_V7 (historical)   : 0.0000 /min
  FAVORABLE set       : [0.0000, 5.9160]
  attainable by model : [0.0000, 1.0430]
  unfavorable reachable: False
  VERDICT             : NON_DISCRIMINATIVE_EXCLUDED

|d_V7 - d_real| = 2.9580 is the largest error attainable by any density in [0.0000, 5.9160]. Every density this model family has ever produced lies in [0.0000, 1.0430], strictly inside that interval, so the rule returns FAVORABLE for every outcome the simulator can reach -- including the outcome in which the candidate produces no spindles at all.


In [6]:

so = power["families"]["so_waveform"]
print("so_waveform")
print("  historical arm verdicts:", so["historical_arm_verdicts"])
print("  VERDICT                :", so["verdict"])
print()
print("admissible:", power["admissible_families"], "| excluded:", power["excluded_families"])


so_waveform
  historical arm verdicts: {'V3 dynamics': 'FAVORABLE', 'V7 event-constrained': 'FAVORABLE', 'V1 canonical (0.8586)': 'UNFAVORABLE'}
  VERDICT                : DISCRIMINATING

admissible: ['so_waveform'] | excluded: ['spindle_density']



### What this costs the closure

The excluded family is the one whose FAVORABLE label turned the primary track's direction
from `UNFAVORABLE` into `MIXED` in the Cursor closure — which under the frozen Notebook 41
mapping is the whole difference between `RELIABILITY_DIAGNOSTIC_FRAMEWORK_ONLY` and
`HELDOUT_DESCRIPTIVE_EVIDENCE_ONLY`. Notebook 41 makes that dependence explicit.



## Phase 3 — Rate precision

Spindle density is a rate. The frozen 120 s protocol, minus 5 s of burn-in, gives the
estimator about two minutes of signal against a reference measured over more than an hour.
The Poisson standard error of that estimate is settled arithmetic.


In [7]:

precision = nb39.phase3_rate_precision(paths)
show(precision, ["simulated_minutes_after_burn_in", "real_reference_minutes",
                 "reference_window_ratio", "expected_events_in_window",
                 "poisson_se_per_min", "relative_se", "required_minutes_for_target",
                 "required_duration_factor", "verdict"])
print()
print(precision["argument"])


{
  "simulated_minutes_after_burn_in": 1.9166666666666667,
  "real_reference_minutes": 71.0,
  "reference_window_ratio": 37.04347826086956,
  "expected_events_in_window": 5.6695,
  "poisson_se_per_min": 1.2422980108758475,
  "relative_se": 0.41997904356857585,
  "required_minutes_for_target": 33.80662609871534,
  "required_duration_factor": 17.63823970367757,
  "verdict": "UNDERPOWERED_AT_FROZEN_BUDGET"
}

At the real rate the frozen window contains 5.67 expected events, giving a Poisson standard error of 1.242/min (42% relative) against a reference measured over 71 minutes. Reaching 10% relative precision needs 33.8 minutes per candidate, 18x the frozen budget. No number of candidates repairs a per-candidate estimator this coarse.



## Phase 4 — The support-QA conflict, adjudicated

The Cursor Notebook 39 recorded `sample_set_difference` on the grounds that no shared
sample-ID alignment artifacts existed. Two things on disk decide it anyway: the per-case
flag rates (restricted to the same estimator) and the two rule implementations, read by AST.

Note what is *not* used as the deciding test. The two rules are nested, so their flag rates
are forced to differ — an equality test would be testing a hypothesis the definitions
already exclude.


In [8]:

support = nb39.phase4_support_conflict(paths)

rates = pd.read_csv(paths["csv"] / "support_conflict_rates.csv")
display(rates)

print("decision conditions:", support["decision_conditions"])
print("cursor label :", support["cursor_nb39_label"])
print("claude labels:", support["claude_labels"])


,track,audit,n_draws,n_flagged,flag_rate,mean_accept_rate,lower_face_tolerance,upper_face_tolerance
0,7d,cursor,1024000,42,0.000041,0.684719,1.000000e-08,0.00001
1,7d,claude,128000,7,0.000055,0.683225,1.000000e-05,0.00001
2,8d,cursor,1024000,41,0.000040,0.690735,1.000000e-08,0.00001
3,8d,claude,128000,11,0.000086,0.685606,1.000000e-05,0.00001


decision conditions: {'nesting_prediction_holds': True, 'exact_atom_counts_agree_at_zero': True, 'acceptance_rates_agree': True}
cursor label : sample_set_difference
claude labels: ['definition_mismatch', 'implementation_mismatch']


In [9]:

print("Cursor rule sites (np.isclose against a face):")
for site in support["rule_evidence"]["cursor_rule"]["isclose_sites"]:
    print("   line %d  vs %.1f" % (site["lineno"], site["compared_to"]))
print(support["rule_evidence"]["cursor_rule"]["explanation"])
print()
print("Claude rule sites (exact equality):")
for site in support["rule_evidence"]["claude_rule"]["exact_equality_sites"]:
    print("   line %d  vs %.1f" % (site["lineno"], site["compared_to"]))
print()
print(support["adjudication"])
print()
print("Does this move NO_GO?", support["no_effect_on_no_go"]["reason"])


Cursor rule sites (np.isclose against a face):
   line 1005  vs 0.0
   line 1005  vs 1.0
   line 1609  vs 0.0
   line 1610  vs 1.0
np.isclose(x, b) tests |x-b| <= atol + rtol*|b|. With the defaults atol=1e-8, rtol=1e-5 this is 1e-8 against the lower face (b=0) but ~1e-5 against the upper face (b=1): a proximity test, asymmetric between the two faces of the same box.

Claude rule sites (exact equality):
   line 555  vs 0.0
   line 555  vs 1.0

The Claude rule flags |x-0| <= 1e-5 or |x-1| <= 1e-5; the Cursor rule flags |x-0| <= 1e-8 or |x-1| <= 1.00001e-5. The Claude flagged set therefore contains the Cursor one up to a 1e-8 sliver at the upper face, and the Claude rate must be the larger of the two -- which is what is observed on both tracks (7d 1.33x, 8d 2.15x). A two-sided equality test of these rates is testing a hypothesis the definitions already exclude, which is why its 8d p-value (0.0212) is not evidence of a sample-set difference. What is diagnostic: the two audits agree exactly


## Phase 5 — Failure-wording ledger

For each upstream failure atom: the sentence the dissertation may write, and the sentence
it may not. This is where the reclassification in Phase 4 actually lands — not in the
verdict, in the prose.


In [10]:

wording = nb39.phase5_wording_ledger(paths, power=power, precision=precision, support=support)
ledger = pd.DataFrame(wording["rows"])
for _, row in ledger.iterrows():
    print("─" * 100)
    print("ATOM      :", row["atom"])
    print("MAY       :", row["may_write"])
    print("MAY NOT   :", row["may_not_write"])


────────────────────────────────────────────────────────────────────────────────────────────────────
ATOM      : S_fail (marginal SBC)
MAY       : Marginal SBC rejected uniformity for 3 of the 7/8 parameters on every estimator after Holm correction.
MAY NOT   : The posterior is globally calibrated for the remaining parameters (non-rejection is not calibration).
────────────────────────────────────────────────────────────────────────────────────────────────────
ATOM      : C_fail / C_lim (coverage)
MAY       : Central coverage at the 90% nominal level fell outside the pre-registered band on the primary track.
MAY NOT   : Coverage was within tolerance on the sensitivity track (C_lim is a limitation flag, not a pass).
────────────────────────────────────────────────────────────────────────────────────────────────────
ATOM      : L_fail (L-C2ST)
MAY       : The local classifier two-sample test rejected at every observation tested.
MAY NOT   : Local failure was caused by the global failure 


## Phase 6 — Handoff


In [11]:

handoff = nb39.phase6_handoff(paths, lineage=lineage, power=power,
                              precision=precision, support=support)
show(handoff)


{
  "created_utc": "2026-08-09T15:37:25.377276+00:00",
  "claude40_handoff_status": "CONTINUE",
  "global_uncertainty_status": "NO_GO",
  "local_analysis_mode": "negative_evidence_only",
  "admissible_families": [
    "so_waveform"
  ],
  "excluded_families": [
    "spindle_density"
  ],
  "exclusion_reasons": {
    "spindle_density": "NON_DISCRIMINATIVE_EXCLUDED"
  },
  "decision_held_out_count": 2,
  "support_conflict_primary_label": "definition_mismatch",
  "rate_precision_verdict": "UNDERPOWERED_AT_FROZEN_BUDGET",
  "budget_plan": {
    "spindle_candidate_runs_dropped": 32,
    "spindle_simulated_seconds_freed": 3840.0,
    "reallocated_to": "so_waveform variance decomposition (R=2)",
    "note": "Repeat 0 of each candidate is the Cursor sprint's persisted trace, reused at zero cost, so only repeats >= 1 are simulated."
  }
}


In [12]:

summary = nb39.run_notebook39_claude()
show(summary)
assert summary["simulations_run"] == 0
assert summary["claude40_handoff_status"] == "CONTINUE"
print("\nNotebook 39 (Claude) complete.")


{
  "created_utc": "2026-08-09T15:37:25.473633+00:00",
  "notebook": "claude_39",
  "global_uncertainty_status": "NO_GO",
  "local_analysis_mode": "negative_evidence_only",
  "admissible_families": [
    "so_waveform"
  ],
  "excluded_families": [
    "spindle_density"
  ],
  "support_conflict_primary_label": "definition_mismatch",
  "cursor_support_label": "sample_set_difference",
  "rate_precision_verdict": "UNDERPOWERED_AT_FROZEN_BUDGET",
  "decision_held_out_count": 2,
  "n_wording_rows": 6,
  "claude40_handoff_status": "CONTINUE",
  "simulations_run": 0,
  "posterior_draws": 0
}

Notebook 39 (Claude) complete.
